In [20]:
import pandas as pd
from scipy.stats import pearsonr, spearmanr
from statsmodels.stats.multitest import multipletests

import matplotlib.pyplot as plt
import seaborn as sns
import os
from datetime import date
import numpy as np
import datetime

pd.options.display.max_rows = 999
pd.options.display.max_columns = 999

In [30]:
df_k9 = pd.read_csv("../H3K9me3-zhuojie-regions/data/zhuojie_c1_avglogfc.csv")
df_k9 = df_k9.rename(columns={"avg_logFC": "logFC_k9"})
df_k9.shape

(91, 5)

In [31]:
df_k9.head(1)

,tissue,cell_type,logFC_k9,num_up_peaks,num_down_peaks
0,brainCB,brainCB-GC_ExN,-1.565036,2,273


In [32]:
df_genes = pd.read_csv("/storage/zhangyanxiaoLab/qihongjian/github/zhanglab-code/projects/paired_seq_tag/code/machine-experiments/RNA/GSEA/data/tpm.csv")
df_genes.shape

(3646332, 6)

In [33]:
df_genes = df_genes[df_genes["cell_type"].isin(df_k9["cell_type"].unique())]
df_genes.shape

(1729676, 6)

In [34]:
df_genes.head()

,proj,gene,tissue,cell_type,mean_TPM_3m,mean_TPM_27m
0,20241110-scPT-Natural-Aging-brainHip-H3K27ac-H...,Gnai3,brainHip,brainHip-Asc_Glia,55.882440,43.297522
1,20241110-scPT-Natural-Aging-brainHip-H3K27ac-H...,Pbsn,brainHip,brainHip-Asc_Glia,5.102051,3.141453
2,20241110-scPT-Natural-Aging-brainHip-H3K27ac-H...,Cdc45,brainHip,brainHip-Asc_Glia,6.220193,6.909143
3,20241110-scPT-Natural-Aging-brainHip-H3K27ac-H...,Scml2,brainHip,brainHip-Asc_Glia,2.004957,3.482241
4,20241110-scPT-Natural-Aging-brainHip-H3K27ac-H...,Apoh,brainHip,brainHip-Asc_Glia,4.209189,1.025006


In [35]:
df_genes[df_genes["gene"]=="Stat5a"].head()

,proj,gene,tissue,cell_type,mean_TPM_3m,mean_TPM_27m
567,20241110-scPT-Natural-Aging-brainHip-H3K27ac-H...,Stat5a,brainHip,brainHip-Asc_Glia,13.348110,12.260705
20352,20241110-scPT-Natural-Aging-brainHip-H3K27ac-H...,Stat5a,brainHip,brainHip-CA1_ExN,0.460335,0.581172
86197,20241110-scPT-Natural-Aging-brainHip-H3K27ac-H...,Stat5a,brainHip,brainHip-CT_SUB_ExN,0.181339,1.321905
119314,20241110-scPT-Natural-Aging-brainHip-H3K27ac-H...,Stat5a,brainHip,brainHip-DG_ExN,1.108278,1.192986
225698,20241110-scPT-Natural-Aging-brainHip-H3K27ac-H...,Stat5a,brainHip,brainHip-InN_Pvalb_InN,1.896894,1.684539


In [36]:
df_k9.head()

,tissue,cell_type,logFC_k9,num_up_peaks,num_down_peaks
0,brainCB,brainCB-GC_ExN,-1.565036,2,273
1,brainCB,brainCB-Ogc_Glia,-1.126710,1,240
2,heart,heart-Mural,-0.878656,0,154
3,lung,lung-capillary_endo,-0.825290,3,231
4,heart,heart-Endocardial_endo,-0.776764,0,170


In [37]:
results = []

for gene, df_g in df_genes.groupby("gene"):
    df_m = (
        df_g
        .merge(df_k9, on=["tissue", "cell_type"], how="outer")
    )

    # missing logFC due to low expression, fill with NA
    df_m["mean_TPM_3m"] = df_m["mean_TPM_3m"].fillna(0)
    
    # 至少需要 10 个点
    if df_m.shape[0] < 10:
        continue

    # Pearson
    r_p, p_p = pearsonr(df_m["mean_TPM_3m"], df_m["logFC_k9"])

    # Spearman
    r_s, p_s = spearmanr(df_m["mean_TPM_3m"], df_m["logFC_k9"])

    results.append({
        "gene": gene,
        "n_points": df_m.shape[0],
        "pearson_r": r_p,
        "pearson_p": p_p,
        "spearman_r": r_s,
        "spearman_p": p_s
    })

/tmp/ipykernel_2709946/3002431862.py:17: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r_p, p_p = pearsonr(df_m["mean_TPM_3m"], df_m["logFC_k9"])
/tmp/ipykernel_2709946/3002431862.py:20: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r_s, p_s = spearmanr(df_m["mean_TPM_3m"], df_m["logFC_k9"])


In [38]:

df_corr = pd.DataFrame(results)
df_corr["pearson_fdr"] = multipletests(df_corr["pearson_p"],  method="fdr_bh")[1]
df_corr["spearman_fdr"] = multipletests(df_corr["spearman_p"],  method="fdr_bh")[1]
df_corr = df_corr.sort_values("pearson_fdr")
df_corr.to_csv("data/H3K9me3_geneTPM_corr.csv", index=False)
df_corr.shape

(31346, 8)

In [39]:
df_corr.head()

,gene,n_points,pearson_r,pearson_p,spearman_r,spearman_p,pearson_fdr,spearman_fdr
0,0610005C13Rik,91,-0.021298,0.841178,0.003536,0.973466,NaN,NaN
1,0610006L08Rik,91,-0.046330,0.662776,0.039773,0.708171,NaN,NaN
2,0610009B22Rik,91,0.067036,0.527807,0.050264,0.636103,NaN,NaN
3,0610009E02Rik,91,-0.212521,0.043129,-0.335579,0.001147,NaN,NaN
4,0610009L18Rik,91,-0.035892,0.735539,-0.162134,0.124671,NaN,NaN


#### plot #####

In [18]:
df_genes["FDR"]=1
pos_corr_f = df_corr[(df_corr['spearman_r'] > 0) & (df_corr['spearman_fdr'] < 0.05)]
neg_corr_f = df_corr[(df_corr['spearman_r'] < 0) & (df_corr['spearman_fdr'] < 0.05)]

# Order genes by spearman_fdr ascending (most significant first)
pos_genes_ordered = pos_corr_f.sort_values('spearman_fdr', ascending=True)['gene'].tolist()
neg_genes_ordered = neg_corr_f.sort_values('spearman_fdr', ascending=True)['gene'].tolist()

# Cell type order: low → high logFC_k9
ct_order = df_k9.sort_values('logFC_k9', ascending=True)['cell_type'].tolist()
logfc_k9_vals = df_k9.set_index('cell_type').loc[ct_order, 'logFC_k9'].values

# Build wide matrices
df_pos_wide = df_genes[df_genes['gene'].isin(pos_genes_ordered)].pivot_table(
    index='gene', columns='cell_type', values='mean_TPM_3m', aggfunc='mean'
).reindex(index=pos_genes_ordered, columns=ct_order)

df_neg_wide = df_genes[df_genes['gene'].isin(neg_genes_ordered)].pivot_table(
    index='gene', columns='cell_type', values='mean_TPM_3m', aggfunc='mean'
).reindex(index=neg_genes_ordered, columns=ct_order)

df_pos_sig = df_genes[df_genes['gene'].isin(pos_genes_ordered)].pivot_table(
    index='gene', columns='cell_type', values='FDR', aggfunc='mean'
).reindex(index=pos_genes_ordered, columns=ct_order)

df_neg_sig = df_genes[df_genes['gene'].isin(neg_genes_ordered)].pivot_table(
    index='gene', columns='cell_type', values='FDR', aggfunc='mean'
).reindex(index=neg_genes_ordered, columns=ct_order)

print(f"Pos: {len(pos_genes_ordered)} genes | Neg: {len(neg_genes_ordered)} genes")
print(f"Top 5 pos genes by FDR: {pos_genes_ordered[:5]}")
print(f"Top 5 neg genes by FDR: {neg_genes_ordered[:5]}")


Pos: 2846 genes | Neg: 1666 genes
Top 5 pos genes by FDR: ['Rhd', 'Tmprss15', 'Slc4a7', 'Tcf20', 'Iqgap2']
Top 5 neg genes by FDR: ['Gm49539', 'Serpinh1', 'S100a16', 'Ptpn21', 'Jup']


In [19]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import numpy as np

vmax = 2.0
bar_colors = ['#D62728' if v > 0 else '#1F77B4' for v in logfc_k9_vals]

def make_full_figure(wide_df, sig_df, genes_ordered, ct_order, logfc_k9_vals,
                     bar_colors, title, ylabel, row_height=0.22, star_fontsize=5):

    n_genes = len(genes_ordered)
    n_ct = len(ct_order)
    fig_height = max(10, n_genes * row_height + 4)

    fig = plt.figure(figsize=(28, fig_height))

    # ✅ 关键：2列 layout，右边给 colorbar
    gs = gridspec.GridSpec(
        2, 2,
        height_ratios=[2.5, n_genes * row_height],
        width_ratios=[50, 1],
        hspace=0.03,
        wspace=0.05
    )

    # ======================
    # 🔵 Bar plot（左上）
    # ======================
    ax_bar = fig.add_subplot(gs[0, 0])

    ax_bar.bar(range(n_ct), logfc_k9_vals,
               color=bar_colors, width=0.8, linewidth=0)

    ax_bar.axhline(0, color='black', linewidth=0.8)
    ax_bar.set_xlim(-0.5, n_ct - 0.5)

    ax_bar.set_xticks([])
    ax_bar.set_ylabel('H3K9me3\nlogFC', fontsize=10)
    ax_bar.tick_params(axis='y', labelsize=8)

    ax_bar.spines['top'].set_visible(False)
    ax_bar.spines['right'].set_visible(False)
    ax_bar.spines['bottom'].set_visible(False)

    ax_bar.set_title(title, fontsize=24, fontweight='bold', pad=10)

    # ======================
    # 🔴 Heatmap（左下）
    # ======================
    ax_heat = fig.add_subplot(gs[1, 0])

    # colorbar 单独轴
    cbar_ax = fig.add_subplot(gs[1, 1])

    mask = wide_df.isna()

    hm = sns.heatmap(
        wide_df.clip(-vmax, vmax),
        mask=mask,
        cmap='RdBu_r',
        center=0,
        vmin=-vmax,
        vmax=vmax,
        ax=ax_heat,
        linewidths=0,
        cbar=True,
        cbar_ax=cbar_ax,   # ✅ 关键
        xticklabels=True,
        yticklabels=True,
    )

    ax_heat.set_facecolor('#D8D8D8')

    # ======================
    # ⭐ 标注显著性
    # ======================
    for i, gene in enumerate(genes_ordered):
        for j, ct in enumerate(ct_order):
            if ct not in sig_df.columns:
                continue

            fdr_val = sig_df.loc[gene, ct]

            if not np.isnan(fdr_val) and fdr_val < 0.05:
                ax_heat.text(
                    j + 0.5, i + 0.5, '*',
                    ha='center', va='center',
                    fontsize=star_fontsize,
                    color='black',
                    fontweight='bold'
                )

    # ======================
    # 🔧 axis 设置
    # ======================
    ax_heat.set_xlabel(
        'Cell type (ordered by H3K9me3 logFC, low → high)',
        fontsize=20
    )
    ax_heat.set_ylabel(ylabel, fontsize=20)

    ax_heat.tick_params(axis='x', labelsize=14, rotation=90)
    ax_heat.tick_params(axis='y', labelsize=14, rotation=0)

    # ======================
    # 🎯 colorbar 美化
    # ======================
    cbar = hm.collections[0].colorbar
    cbar.set_label('Aging gene logFC\n(clipped ±2)', fontsize=9)
    cbar.ax.tick_params(labelsize=7)

    return fig


# ======================
# 🚀 Figure 1: Positive
# ======================

fig1 = make_full_figure(
    df_pos_wide,
    df_pos_sig,
    pos_genes_ordered,
    ct_order,
    logfc_k9_vals,
    bar_colors,
    title='Aging genes positively correlated with H3K9me3 aging\n(Spearman r > 0, FDR < 0.05)',
    ylabel='Gene (ordered by Spearman FDR, low → high)',
)

fig1.show()


# ======================
# 🚀 Figure 2: Negative
# ======================

fig2 = make_full_figure(
    df_neg_wide,
    df_neg_sig,
    neg_genes_ordered,
    ct_order,
    logfc_k9_vals,
    bar_colors,
    title='Aging genes negatively correlated with H3K9me3 aging\n(Spearman r < 0, FDR < 0.05)',
    ylabel='Gene (ordered by Spearman FDR, low → high)',
)

fig2.show()